<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity/blob/main/notebooks/02_select_lakes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 - Choose the target lakes, and measure the ceiling

Region: **Northern Lower Michigan** (clear glacial kettle lakes, same optical regime as the
New Hampshire reference lakes, with decades of Cooperative Lakes Monitoring Program Secchi).

**The gate.** One large lake and one small one, both with enough FIELD July-years to support a
client-style July-annual-mean validation. Selection is on distinct years with a July field
Secchi reading in the Water Quality Portal, NOT on coincident satellite/in-situ matchups.
Matchups require a clean pass within three days of a reading and badly undercount the
achievable validation sample; the client's own r = -0.22 for Squam used the full field record,
not matchups. If no small lake qualifies, `select_target_lakes` raises.

**The ceiling.** The intraclass correlation of log Secchi across the region's lakes: the
fraction of the pooled signal that says nothing about tracking one lake through time. It is the
number that reconciles a published R-squared of 0.637 with a per-lake r of -0.22.

In [ ]:
# Cell 1 of 6: repo + Drive
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))  # make lakeclarity importable in the running kernel

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

In [ ]:
# Cell 2 of 6: region subset (training matchups)
import pandas as pd
from lakeclarity import config, eda, edi, features, locus, selection, viz, wqp

viz.use_style()

matchups = edi.load_matchups()
lakes = locus.load_lakes()
region_lk = locus.region_lakes(lakes)
print(f"{len(region_lk):,} {config.REGION_NAME} lakes in LOCUS")

region = matchups[matchups["lagoslakeid"].isin(region_lk["lagoslakeid"])].copy()
region = region[region[config.TARGET].notna()]
region = features.add_time_columns(region)
region = features.add_log_target(region)
print(f"{len(region):,} Secchi matchups on {region.lagoslakeid.nunique():,} lakes (training supply)")
region.to_parquet(config.INTERIM_DIR / "region_matchups.parquet", index=False)

In [ ]:
# Cell 3 of 6: FIELD coverage from the Water Quality Portal (the selection metric)
# Confirm the characteristic-name choice, then pull field Secchi + station coords,
# map each station to its nearest lake, and count distinct July-years per lake.
print(wqp.characteristic_coverage().to_string(index=False), "\n")

field = wqp.fetch_secchi()                 # cached to Drive after first pull
stations = wqp.fetch_stations()            # site coordinates
site_to_lake = wqp.map_sites_to_lakes(stations, lakes)
field_cov = wqp.lake_field_coverage(field, site_to_lake)

site_to_lake.to_parquet(config.INTERIM_DIR / "wqp_site_to_lake.parquet", index=False)
field_cov.to_csv(config.TABLE_DIR / "T04b_field_coverage.csv")
print(f"{len(field):,} field Secchi records, {len(stations):,} stations, "
      f"{len(site_to_lake):,} mapped to a lake")
print(f"lakes with >= {config.MIN_FIELD_JULY_YEARS} field July-years: "
      f"{(field_cov['field_july_years'] >= config.MIN_FIELD_JULY_YEARS).sum()}")
print(field_cov.head(12).to_string())

In [ ]:
# Cell 4 of 6: THE CEILING. Run this before choosing anything.
ceiling = selection.ceiling_report(region)
for k, v in ceiling.items():
    print(f"{k:>28}: {v:,.3f}")

print(f"\nRead this as: of all the variation in lake clarity across {config.REGION_NAME},")
print(f"{ceiling['pct_variance_between_lakes']:.0f}% is lakes differing from one another and only")
print(f"{ceiling['pct_variance_within_lakes']:.0f}% is any given lake changing across the years.")
print("\nA model that learns the first and none of the second scores a superb pooled R2")
print("and cannot track a single lake. That is the national model's failure mode.")

pd.Series(ceiling).to_csv(config.TABLE_DIR / "T03_variance_ceiling.csv")

In [ ]:
# Cell 5 of 6: the gate
candidates = selection.candidate_table(region, region_lk, field_cov)
candidates.to_csv(config.TABLE_DIR / "T04_candidate_lakes.csv", index=False)
print(candidates.head(15)[["lagoslakeid", "lake_name", "lake_waterarea_ha",
                            "field_july_years", "field_year_first", "field_year_last",
                            "field_secchi_mean", "n_matchups"]].to_string(index=False))

try:
    targets = selection.select_target_lakes(candidates)
    print("\n" + targets.summary())
    pd.Series({"large": targets.ids[0], "small": targets.ids[1]}).to_csv(
        config.TABLE_DIR / "T05_target_lakes.csv")
except selection.NoEligibleLakeError as exc:
    print("\nGATE FAILED:", exc)
    print("\nDo not work around this. Either relax MIN_FIELD_JULY_YEARS with a written")
    print("justification, widen the region, or accept that the small-lake case cannot")
    print("be validated with the data that exists.")
    raise

In [ ]:
# Cell 6 of 6: figures F6 to F9
for fig, fid, slug in [
    (selection.fig_region_map(candidates, targets), "F06", "region_map"),
    (selection.fig_area_vs_pixels(candidates, targets), "F07", "area_vs_pixels"),
    (selection.fig_variance_decomposition(region), "F08", "variance_decomposition"),
    (selection.fig_candidate_timeseries(field, site_to_lake, candidates), "F09", "candidate_timeseries"),
]:
    print("wrote", viz.save(fig, fid, slug).name)